# Milestone 1: Exploratory Data Analysis
Heart Disease Risk Prediction (BRFSS 2022)

Covers the September milestone:
- feature typing / encoding decisions
- missing data audit
- correlation matrix + target relationships
- distributions, bivariate analysis, summary stats
- fixed-seed, stratified train/validation/test split


In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

RANDOM_SEED = 42  # fixed seed used everywhere below for reproducibility

In [3]:
DATA_PATH = "../Data/heart_2022_with_nans.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(2, 1)


,version https://git-lfs.github.com/spec/v1
0,oid sha256:9c5be2d641c60b63d395cb374581e4a691d...
1,size 139870876


## 2. Feature typing
Document each column's storage dtype and its semantic type (numerical continuous,
numerical discrete, binary categorical, multi-class categorical, free text).
This will help determine how to encode it later.

In [4]:
dtype_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unique': df.nunique(),
    'sample_values': [df[c].dropna().unique()[:5] for c in df.columns]
})
dtype_summary

,dtype,n_unique,sample_values
version https://git-lfs.github.com/spec/v1,object,2,[oid sha256:9c5be2d641c60b63d395cb374581e4a691...


# TODO: after reviewing dtype_summary above, classify each column
### Data Types

| Column | Data Type |
|---|---|
| State | Categorical (String) |
| Sex | Categorical (Binary) |
| GeneralHealth | Categorical (Ordinal) |
| PhysicalHealthDays | Numerical (Float) |
| MentalHealthDays | Numerical (Float) |
| LastCheckupTime | Categorical (Ordinal) |
| PhysicalActivities | Categorical (Binary) |
| SleepHours | Numerical (Float) |
| RemovedTeeth | Categorical (Ordinal) |
| HadHeartAttack | Categorical (Binary) |
| HadAngina | Categorical (Binary) |
| HadStroke | Categorical (Binary) |
| HadAsthma | Categorical (Binary) |
| HadSkinCancer | Categorical (Binary) |
| HadCOPD | Categorical (Binary) |
| HadDepressiveDisorder | Categorical (Binary) |
| HadKidneyDisease | Categorical (Binary) |
| HadArthritis | Categorical (Binary) |
| HadDiabetes | Categorical (Nominal) |
| DeafOrHardOfHearing | Categorical (Binary) |
| BlindOrVisionDifficulty | Categorical (Binary) |
| DifficultyConcentrating | Categorical (Binary) |
| DifficultyWalking | Categorical (Binary) |
| DifficultyDressingBathing | Categorical (Binary) |
| DifficultyErrands | Categorical (Binary) |
| SmokerStatus | Categorical (Nominal) |
| ECigaretteUsage | Categorical (Nominal) |
| ChestScan | Categorical (Binary) |
| RaceEthnicityCategory | Categorical (Nominal) |
| AgeCategory | Categorical (Ordinal) |
| HeightInMeters | Numerical (Float) |
| WeightInKilograms | Numerical (Float) |
| BMI | Numerical (Float) |
| AlcoholDrinkers | Categorical (Binary) |
| HIVTesting | Categorical (Binary) |
| FluVaxLast12 | Categorical (Binary) |
| PneumoVaxEver | Categorical (Binary) |
| TetanusLast10Tdap | Categorical (Nominal) |
| HighRiskLastYear | Categorical (Binary) |
| CovidPos | Categorical (Nominal) |

## 3. Missing data audit
For each column: how many values are missing, and what's a defensible handling strategy
(drop rows, impute mean/median/mode, impute with a model, or add a "missing" indicator
category)?

In [5]:
missing = df.isna().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({'n_missing': missing})
missing_summary[missing_summary['n_missing'] > 0]

,n_missing


In [6]:
missing_summary[missing_summary['n_missing'] == 0]

,n_missing
version https://git-lfs.github.com/spec/v1,0


In [8]:
# TODO: document your per-column decision

# Sex and state do not have missing values.

# HighRiskLastYear is our target variable and has 50623 missing values.
# These rows will be removed since they will not be helpful in our analysis.

# Missing rows in HadHeartAttack will also be removed since the variable
# directly involves heart risks

df = df[df['HighRiskLastYear'].notna()]
df = df[df['HadHeartAttack'].notna()]

# Check how many more NA values drop after removing those missing rows
print(df.shape)
missing_updated = df.isna().sum().sort_values(ascending=False)
missing_summary_updated = pd.DataFrame({'n_missing': missing_updated})
missing_summary_updated[missing_summary_updated['n_missing'] >= 0]

KeyError: 'HighRiskLastYear'

In [ ]:
# Displaying percentages of missing values
missing_counts_perc = df.isna().sum().sort_values(ascending=False)
missing_percentages = (df.isna().mean() * 100).round(2)
missing_percentages_summary = pd.DataFrame({'n_missing_%': missing_percentages})
missing_percentages[missing_percentages_summary['n_missing_%'] > 0].sort_values(ascending=False)

TetanusLast10Tdap            9.11
PneumoVaxEver                7.93
BMI                          6.65
WeightInKilograms            5.30
HIVTesting                   4.80
ChestScan                    4.22
RaceEthnicityCategory        2.79
RemovedTeeth                 2.34
PhysicalHealthDays           2.28
HeightInMeters               2.21
MentalHealthDays             1.89
LastCheckupTime              1.73
AgeCategory                  1.64
AlcoholDrinkers              1.24
SleepHours                   1.10
HadAngina                    0.81
FluVaxLast12                 0.73
SmokerStatus                 0.71
DifficultyConcentrating      0.64
HadSkinCancer                0.63
HadDepressiveDisorder        0.53
HadArthritis                 0.51
CovidPos                     0.41
HadCOPD                      0.40
ECigaretteUsage              0.37
HadKidneyDisease             0.35
DeafOrHardOfHearing          0.35
DifficultyWalking            0.34
HadAsthma                    0.32
DifficultyErra

In [ ]:
# Each column's missing values take up less than 10% of its rows, majority being under 3%.
# Therefore, we do not need to drop any columns.

# Now, we can perform a stratisfied split on the dataset.
X = df.drop(columns = ['HighRiskLastYear'])
y = df['HighRiskLastYear']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

# Check shapes after splitting
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}\n")
print(f"Target positive rate in original data: {y.value_counts(normalize=True)}\n")
print(f"Target positive rate in training set: {y_train.value_counts(normalize=True)}\n")
print(f"Target positive rate in test set: {y_test.value_counts(normalize=True)}")

X_train: (313665, 39)
y_train: (313665,)
X_test: (78417, 39)
y_test: (78417,)

Target positive rate in original data: HighRiskLastYear
No     0.956422
Yes    0.043578
Name: proportion, dtype: float64

Target positive rate in training set: HighRiskLastYear
No     0.956422
Yes    0.043578
Name: proportion, dtype: float64

Target positive rate in test set: HighRiskLastYear
No     0.956425
Yes    0.043575
Name: proportion, dtype: float64


In [ ]:
# Because we are using XGBoost, the model is able to handle NA values.
# We can explore how well utilizing this algorithm will impact our model
# and revise EDA and data cleaning steps as needed.

## 4. Correlation matrix
Numeric-only Pearson correlation first, then rank features by their correlation with the
target. For categorical predictors, a full correlation matrix isn't meaningful as-is --
consider encoding them before including them.

In [ ]:
# Encode target, then rank features by correlation with it
target_col = 'HadHeartAttack'

## 5. Distributions, bivariate analysis, and summary statistics

In [ ]:
df[numeric_cols].describe().T  # mean, std, min, max, quartiles for every numeric column

,count,mean,std,min,25%,50%,75%,max
PhysicalHealthDays,434205.0,4.347919,8.688912,0.00,0.00,0.00,3.00,30.00
MentalHealthDays,436065.0,4.382649,8.387475,0.00,0.00,0.00,5.00,30.00
SleepHours,439679.0,7.022983,1.502425,1.00,6.00,7.00,8.00,24.00
HeightInMeters,416480.0,1.702691,0.107177,0.91,1.63,1.70,1.78,2.41
WeightInKilograms,403054.0,83.074470,21.448173,22.68,68.04,80.74,95.25,292.57
BMI,396326.0,28.529842,6.554889,12.02,24.13,27.44,31.75,99.64


In [ ]:
# Univariate: distribution of each numeric feature

In [ ]:
# Bivariate: each numeric feature vs. target

In [ ]:
# Bivariate for categorical features

## 6. Train / validation / test split

In [ ]:
# First split off the test set, then split train/val from what remains